# 02. Redis Deep Dive — 8가지 패턴 완전 정복

`난이도: 중급` · `소요: 60분` · `사전 지식:` [01. 프로젝트 개요](./01_project_overview.ipynb) `완료`

---

## 목차
1. [환경 설정](#1.-환경-설정)
2. [Redis 개요 — 왜 Redis인가?](#2.-Redis-개요)
3. [Pattern 1: Pub/Sub](#3.-Pattern-1:-Pub/Sub) — 방송국처럼 메시지를 뿌린다
4. [Pattern 2: Stream](#4.-Pattern-2:-Stream) — 놓친 방송도 다시 볼 수 있다
5. [Pattern 3: List Queue](#5.-Pattern-3:-List-Queue) — 은행 번호표 시스템
6. [Pattern 4: Cache](#6.-Pattern-4:-Cache) — 자주 찾는 건 책상 위에 둔다
7. [Pattern 5: Rate Limiter](#7.-Pattern-5:-Rate-Limiter) — 놀이공원 입장 제한
8. [Pattern 6: Bloom Filter](#8.-Pattern-6:-Bloom-Filter) — 억 단위 URL 중복 체크, 메모리 1%로
9. [Pattern 7: TimeSeries](#9.-Pattern-7:-TimeSeries) — 시간 기반 메트릭 저장소
10. [Pattern 8: Vector Set](#10.-Pattern-8:-Vector-Set) — AI 시맨틱 검색 (Redis 8.0+)
11. [관리 명령어 실습](#11.-aioredis-직접-실습)
12. [정리 및 요약](#12.-정리-및-요약-테이블)

---

### 학습 목표
- Redis가 "캐시"만이 아니라 **8가지 완전히 다른 역할**을 할 수 있다는 것을 이해
- 각 패턴이 **어떤 문제를 해결하는지** 실생활 비유로 파악
- API 호출과 Redis 직접 명령, 두 가지 경로로 실습
- Redis 8.0 전용 기능 (Vector Set) 포함 최신 패턴 실습

> **Tip** — 이 노트북은 FastAPI 서버를 통한 API 호출과 Redis에 직접 명령을 보내는 두 가지 방식을 모두 사용합니다.
> 두 방식의 차이를 비교하면서 학습하면 이해가 깊어집니다.

> **관련 노트북** — 고급 Redis 패턴(Lua Scripting, MULTI/EXEC 트랜잭션)은 [07. 고급 패턴](./07_advanced_patterns.ipynb)에서 다룹니다.

---

#### 노트북 네비게이션
| 이전 | 현재 | 다음 |
|------|------|------|
| [01. 프로젝트 개요](./01_project_overview.ipynb) | **02. Redis Deep Dive** | [03. RabbitMQ Deep Dive](./03_rabbitmq_deep_dive.ipynb) |

---
## 1. 환경 설정

노트북은 `notebooks/` 폴더에서 실행되므로, 프로젝트 루트를 `sys.path`에 추가해야
`app` 모듈을 정상적으로 import할 수 있습니다.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = str(Path.cwd().parent)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import httpx
BASE_URL = "http://localhost:8000"
client = httpx.AsyncClient(base_url=BASE_URL, timeout=10.0)

`aioredis`는 API 서버를 거치지 않고 Redis에 직접 명령을 보낼 때 사용합니다.
이를 통해 내부 동작을 눈으로 확인할 수 있습니다.

In [ ]:
import redis.asyncio as aioredis
import json, time

r = aioredis.from_url("redis://localhost:6379", decode_responses=True)
pong = await r.ping()
print(f"Redis 직접 연결: {pong}")

---
## 2. Redis 개요 — 왜 Redis인가?

### Redis를 한 마디로 설명하면

> "메모리에 살고 있는 만능 데이터 구조 서버"

대부분의 사람들은 Redis를 "캐시"로만 알고 있습니다. 하지만 Redis는 **자료구조(data structure) 서버**입니다.
마치 Python에 `list`, `dict`, `set`이 있는 것처럼, Redis에도 여러 자료구조가 있고,
각 자료구조가 **서로 다른 문제를 해결**합니다.

### 실생활 비유로 이해하기

| Redis 자료구조 | 실생활 비유 | 메시징 패턴 | 핵심 질문 |
|---------------|------------|------------|----------|
| **Channel** (Pub/Sub) | 라디오 방송 — 지금 듣고 있어야만 들을 수 있음 | 실시간 브로드캐스트 | "지금 이 순간 누가 듣고 있나?" |
| **Stream** | YouTube 라이브+다시보기 — 놓쳐도 나중에 볼 수 있음 | 이벤트 로그 | "이 이벤트를 누가 처리했나?" |
| **List** | 은행 번호표 — 먼저 온 사람이 먼저 처리됨 | 작업 큐 | "다음 처리할 작업이 뭐지?" |
| **String** (+ TTL) | 포스트잇 메모 — 일정 시간 후 자동으로 떨어짐 | 캐시 | "이 데이터 아직 유효한가?" |
| **Sorted Set** | 놀이공원 입장 카운터 — 시간대별 방문자 수 추적 | 속도 제한 | "이 사용자가 너무 자주 요청하나?" |

### 왜 Redis가 빠른가?

```text
일반 DB (PostgreSQL, MySQL):
  클라이언트 → [네트워크] → [디스크에서 읽기] → [결과 반환]
                              ↑ 여기가 느림 (ms 단위)

Redis:
  클라이언트 → [네트워크] → [메모리에서 읽기] → [결과 반환]
                              ↑ 여기가 빠름 (μs 단위, 1000배 차이)
```

Redis는 모든 데이터를 **RAM(메모리)**에 저장합니다.
디스크 I/O가 없으므로 응답 시간이 마이크로초(μs) 단위입니다.
대신 메모리 용량 제한이 있으므로, **자주 접근하는 소량의 데이터**에 적합합니다.

### 이 프로젝트에서 Redis의 역할

```text
                   ┌─ [1] Pub/Sub: 실시간 알림 전송
                   ├─ [2] Stream:  이벤트 로그 저장 + 병렬 처리
FastAPI 서버 ──→ Redis ├─ [3] Queue:   백그라운드 작업 등록
                   ├─ [4] Cache:   API 응답 캐싱 (DB 부하 감소)
                   └─ [5] Rate Limiter: API 호출 횟수 제한
```

> **핵심** — Pub/Sub와 Stream의 차이가 이 노트북에서 가장 중요한 개념입니다.
> "메시지를 저장하느냐, 안 하느냐"가 두 패턴을 나누는 결정적 기준입니다.

### Redis 8 — 무엇이 달라졌나?

이 프로젝트는 **Redis 8**을 사용합니다. Redis 7 이하와의 주요 차이점:

| 변경사항 | 설명 |
|----------|------|
| **Redis Stack 통합** | JSON, Search, TimeSeries, Bloom 모듈이 코어에 내장. 별도 `redis-stack` 이미지 불필요 |
| **Hash 신규 명령** | `HGETDEL` (GET+DEL 원자적), `HSETEX` (SET+TTL), `HGETEX` (GET+TTL 갱신) |
| **Vector Set** | AI/ML 벡터 유사도 검색 네이티브 지원 (`VADD`, `VSIM`) |
| **메모리 최적화** | Hash/Sorted Set 메모리 사용량 대폭 감소, 최대 87% 빠른 명령 처리 |
| **Stream 멱등성** | 동일 ID로 중복 추가 시 자동 무시 (exactly-once 보장 강화) |

> **왜 Redis 8인가?** — 학습 관점에서 기존 5가지 패턴(Pub/Sub, Stream, Queue, Cache, Rate Limiter)은
> 동일하게 동작하면서, Redis Stack 모듈을 추가 설치 없이 바로 사용할 수 있습니다.

---
## 3. Pattern 1: Pub/Sub

### 라디오 방송과 같은 원리

Pub/Sub를 이해하는 가장 쉬운 방법은 **라디오 방송**을 떠올리는 것입니다.

```text
        ┌─────────────────────────────────────────────────┐
        │              Redis Pub/Sub 구조                   │
        │                                                   │
        │  방송국(Publisher)                                  │
        │     │                                             │
        │     ▼                                             │
        │  ┌───────────────┐                                │
        │  │  "news" 채널   │ ◄── 하나의 주파수(채널)         │
        │  └───┬───┬───┬───┘                                │
        │      │   │   │                                    │
        │      ▼   ▼   ▼                                    │
        │     A   B   C   ◄── 지금 듣고 있는 사람만 수신     │
        │                                                   │
        │  라디오를 안 켠 사람 D? → 영원히 못 들음!           │
        └─────────────────────────────────────────────────┘
```

**핵심 특성을 정리하면:**

1. **실시간 전달**: 메시지를 보내면 **지금 구독 중인 클라이언트에게만** 즉시 전달됩니다
2. **저장 안 됨**: 메시지는 어디에도 저장되지 않습니다. 구독자가 0명이면 메시지는 그냥 사라집니다
3. **수신 확인 불가**: 발행자는 "누가 받았는지"만 알고, "제대로 처리했는지"는 모릅니다

### 언제 쓰고, 언제 쓰면 안 되나?

| 적합한 경우 | 부적합한 경우 |
|------------|-------------|
| 채팅방 메시지 전달 | 주문 처리 (유실되면 큰일) |
| 실시간 주가 업데이트 | 결제 요청 (한 건도 빠지면 안 됨) |
| 게임 이벤트 알림 | 이메일 발송 큐 (나중에라도 보내야 함) |
| 캐시 무효화 신호 | 로그 수집 (놓치면 분석 불가) |

> **한 줄 요약** — "지금 안 들으면 영원히 못 듣는 라디오 방송"
> 유실되어도 괜찮은 실시간 데이터에만 사용하세요.

### 소스코드 분석: `RedisBroker.publish()`

`publish()` 메서드는 `AbstractBroker`의 필수 구현입니다.
내부적으로 Redis의 `PUBLISH` 명령을 호출하며, 반환값으로 **수신한 구독자 수**를 알려줍니다.

In [14]:
import inspect
from app.brokers.redis_broker import RedisBroker

src = inspect.getsource(RedisBroker.publish)
print(src)

    @measure_time(broker="redis", operation="publish")
    async def publish(self, destination: str, message: dict) -> dict:
        """Pub/Sub: 메시지 발행"""
        payload = json.dumps({
            **message,
            "timestamp": time.time(),
            "broker": self.name,
            "pattern": "pubsub",
        })
        subscribers = await self.client.publish(destination, payload)

        return {
            "broker": self.name,
            "pattern": "pubsub",
            "destination": destination,
            "subscribers_received": subscribers,
        }



### API 테스트: Pub/Sub 메시지 발행

구독자가 없는 상태에서 발행하면 `subscribers_received`가 0입니다.
이것이 Pub/Sub의 핵심 한계 -- 수신 확인이 불가능하고, 유실 여부를 알 수 없습니다.

In [15]:
resp = await client.post("/redis/pubsub/publish", json={
    "content": "Pub/Sub 테스트 메시지",
    "metadata": {"notebook": "02", "pattern": "pubsub"}
})
result = resp.json()
print(json.dumps(result, indent=2, ensure_ascii=False))

{
  "broker": "redis",
  "pattern": "pubsub",
  "destination": "test-channel",
  "subscribers_received": 0,
  "elapsed_ms": 6.278
}


### aioredis 직접 명령: PUBLISH

API를 거치지 않고 Redis에 직접 `PUBLISH` 명령을 보내봅니다.
반환값은 메시지를 수신한 구독자 수(정수)입니다.

In [16]:
payload = json.dumps({"msg": "직접 발행 테스트", "ts": time.time()})
receivers = await r.publish("direct-channel", payload)
print(f"수신한 구독자 수: {receivers}")
print("-> 구독자가 없으므로 0, 메시지는 유실됨")

수신한 구독자 수: 0
-> 구독자가 없으므로 0, 메시지는 유실됨


### Pub/Sub 패턴 요약

| 항목 | 설명 |
|------|------|
| Redis 명령 | `PUBLISH channel message` / `SUBSCRIBE channel` |
| 메시지 영속성 | X -- 수신자 없으면 유실 |
| 메시지 순서 보장 | O -- 단일 채널 내 FIFO |
| 적합한 사용처 | 실시간 알림, 채팅, 이벤트 브로드캐스트 |
| 부적합한 사용처 | 주문 처리, 결제 등 유실 불가 시나리오 |
| API 엔드포인트 | `POST /redis/pubsub/publish` |

---
## 4. Pattern 2: Stream (Redis 5.0+)

### YouTube 다시보기와 같은 원리

Pub/Sub가 "라디오 방송"이라면, Stream은 **"YouTube 라이브 + 다시보기"**입니다.

```text
  Pub/Sub (라디오):                    Stream (YouTube):
  ┌─────────────────┐                 ┌──────────────────────────┐
  │                  │                 │  저장됨! (다시보기 가능)   │
  │ "안녕하세요" ──→ 사라짐!          │  ID       내용            │
  │  (구독자 없으면)   │                 │  1700-0   "첫 번째 이벤트" │
  │                  │                 │  1700-1   "두 번째 이벤트" │
  └─────────────────┘                 │  1700-2   "세 번째 이벤트" │
                                      │  ...      (계속 쌓임)     │
                                      └──────────────────────────┘
```

### Pub/Sub vs Stream: 결정적 차이 3가지

| | Pub/Sub | Stream |
|---|---------|--------|
| **저장** | X — 보내고 끝 | O — 모든 메시지에 ID가 붙어 저장됨 |
| **나중에 읽기** | 불가능 | 가능 — ID로 과거 메시지 조회 |
| **처리 확인** | 없음 | XACK로 "이 메시지 처리 완료" 표시 |

### Consumer Group: 팀으로 나눠 처리하기

혼자서 모든 메시지를 처리하면 느립니다. **Consumer Group**은 팀원들이 메시지를 **나눠 받는** 구조입니다.

```text
  Stream에 메시지가 5개 쌓여있다고 가정:

  Stream: [이벤트A] [이벤트B] [이벤트C] [이벤트D] [이벤트E]

  ── Consumer Group "주문팀" ─────────────────────────
  │
  │  워커1(김직원) → 이벤트A, 이벤트C, 이벤트E를 담당
  │  워커2(박직원) → 이벤트B, 이벤트D를 담당
  │
  │  → 하나의 메시지는 팀 내 한 명만 처리! (중복 처리 없음)
  │  → 워커1이 죽으면? 미처리 메시지가 PEL에 남아 다른 워커가 이어받음
  ──────────────────────────────────────────────────

  ── Consumer Group "분석팀" (별개의 그룹) ──────────
  │
  │  분석워커1 → 이벤트A~E 전부 독립적으로 읽음
  │  → 다른 그룹과 간섭 없이 같은 데이터를 각자 처리
  ──────────────────────────────────────────────────
```

**핵심 포인트:**
- **같은 그룹 내** → 메시지를 **분배** (한 메시지는 한 워커만)
- **다른 그룹 간** → 메시지를 **복제** (모든 그룹이 전체를 읽음)
- **PEL (Pending Entries List)** → ACK 안 된 메시지 목록. 워커 장애 시 다른 워커가 재처리 가능

### XACK가 중요한 이유

```text
  워커가 메시지를 읽음
       │
       ▼
  ┌── 처리 성공 ──→ XACK 호출 → PEL에서 제거 (완료!)
  │
  └── 처리 중 서버 다운 ──→ XACK 안 됨 → PEL에 남아있음
                                          → 다른 워커가 다시 처리 가능
```

> **한 줄 요약** — "메시지가 저장되고, 팀으로 나눠 처리하며, 실패하면 재시도할 수 있는 이벤트 로그"
> Kafka와 비슷하지만, 단일 서버에서 간단하게 쓸 수 있다는 것이 장점입니다.

### Stream vs Pub/Sub: 실전 선택 기준

위에서 개념을 배웠으니, 실전에서 어떤 걸 선택해야 하는지 정리해봅시다.

```text
질문 1: "메시지를 잃어버려도 괜찮나요?"
  ├── YES → Pub/Sub (더 간단하고 빠름)
  └── NO  → 질문 2로...

질문 2: "여러 워커가 메시지를 나눠 처리해야 하나요?"
  ├── YES → Stream + Consumer Group
  └── NO  → Stream (단일 소비자)
```

> 다음 코드 셀에서 이 두 패턴의 차이를 직접 확인해봅시다.

### 소스코드 분석: `stream_add()` (XADD)

`XADD`는 Stream에 이벤트를 추가합니다. 반환값은 자동 생성된 메시지 ID입니다.
`xlen()`으로 현재 스트림 길이도 함께 반환하여 상태를 파악할 수 있게 합니다.

In [ ]:
src = inspect.getsource(RedisBroker.stream_add)
print(src)

    @measure_time(broker="redis_stream", operation="publish")
    async def stream_add(self, stream: str, message: dict) -> dict:
        """Stream: 이벤트 추가 (XADD)"""
        entry = {
            "data": json.dumps(message),
            "broker": self.name,
            "pattern": "stream",
            "produced_at": str(time.time()),
        }
        msg_id = await self.client.xadd(stream, entry)
        length = await self.client.xlen(stream)

        return {
            "broker": self.name,
            "pattern": "stream",
            "destination": stream,
            "message_id": msg_id,
            "stream_length": length,
        }



### API 테스트: Stream에 이벤트 추가

Stream에 메시지를 추가하면 `message_id`와 `stream_length`가 반환됩니다.
ID 형식은 `1234567890123-0` (밀리초 타임스탬프-시퀀스)입니다.

In [18]:
for i in range(3):
    resp = await client.post("/redis/stream/add", json={
        "content": f"이벤트 #{i+1}",
        "metadata": {"seq": i+1}
    })
    result = resp.json()
    print(f"#{i+1} id={result['message_id']}, length={result['stream_length']}")

#1 id=1771384042402-0, length=4
#2 id=1771384042416-0, length=5
#3 id=1771384042417-0, length=6


### API 테스트: Stream 읽기 (XREAD)

`XREAD`는 특정 ID 이후의 메시지를 읽습니다. `last_id=0`이면 처음부터 읽습니다.
Pub/Sub와 달리 **이미 발행된 메시지도 읽을 수 있습니다.**

In [19]:
resp = await client.get("/redis/stream/read", params={
    "count": 10, "last_id": "0"
})
result = resp.json()
print(f"읽은 메시지 수: {result['count']}")
for msg in result["messages"]:
    print(f"  {msg['id']}: {msg['data']}")

읽은 메시지 수: 6
  1771379629202-0: {'content': '이벤트 #1', 'seq': 1}
  1771379629208-0: {'content': '이벤트 #2', 'seq': 2}
  1771379629210-0: {'content': '이벤트 #3', 'seq': 3}
  1771384042402-0: {'content': '이벤트 #1', 'seq': 1}
  1771384042416-0: {'content': '이벤트 #2', 'seq': 2}
  1771384042417-0: {'content': '이벤트 #3', 'seq': 3}


### Consumer Group 개념

Consumer Group은 Stream의 핵심 기능입니다.
같은 그룹 내의 Consumer들은 메시지를 **분배**받습니다 (한 메시지는 한 Consumer만 처리).
다른 그룹은 **독립적으로** 같은 메시지를 모두 읽을 수 있습니다.

```text
Stream: [A, B, C, D, E]
  Group-1:
    consumer-1 → A, C, E  (분배)
    consumer-2 → B, D
  Group-2:
    consumer-3 → A, B, C, D, E  (독립)
```


### API 테스트: Consumer Group 생성 및 읽기

`XGROUP CREATE`로 그룹을 만들고, `XREADGROUP`으로 읽습니다.
읽은 메시지는 자동으로 `XACK` 처리되어 재전달되지 않습니다.

In [20]:
# Consumer Group 생성
resp = await client.post("/redis/stream/group/create", params={
    "group": "notebook-group"
})
print("그룹 생성:", resp.json())

그룹 생성: {'status': 'created', 'stream': 'test-stream', 'group': 'notebook-group'}


In [21]:
# Consumer Group으로 읽기 (consumer-1)
resp = await client.get(
    "/redis/stream/group/read",
    params={"group": "notebook-group", "consumer": "consumer-1", "count": 5}
)
result = resp.json()
print(f"consumer-1이 읽은 메시지: {result['count']}개")
for msg in result["messages"]:
    print(f"  {msg['id']}: {msg['data']}")

consumer-1이 읽은 메시지: 5개
  1771379629202-0: {'content': '이벤트 #1', 'seq': 1}
  1771379629208-0: {'content': '이벤트 #2', 'seq': 2}
  1771379629210-0: {'content': '이벤트 #3', 'seq': 3}
  1771384042402-0: {'content': '이벤트 #1', 'seq': 1}
  1771384042416-0: {'content': '이벤트 #2', 'seq': 2}


### 소스코드 분석: XACK 자동 처리

`stream_read_group()` 내부에서 메시지를 읽은 직후 `xack()`를 호출합니다.
XACK를 하지 않으면 메시지가 **PEL(Pending Entries List)**에 남아,
장애 시 다른 Consumer가 재처리할 수 있습니다.

In [23]:
src = inspect.getsource(RedisBroker.stream_read_group)
print(src)

    @measure_time(broker="redis_stream", operation="consume", record_metric=False)
    async def stream_read_group(
        self, stream: str, group: str, consumer: str, count: int = 10
    ) -> dict:
        """Stream: Consumer Group으로 읽기 (XREADGROUP)"""
        messages = await self.client.xreadgroup(
            group, consumer, {stream: ">"}, count=count
        )

        parsed = []
        if messages:
            for _stream_name, entries in messages:
                for msg_id, fields in entries:
                    parsed.append({
                        "id": msg_id,
                        "data": json.loads(fields.get("data", "{}")),
                        "produced_at": fields.get("produced_at"),
                    })
                    await self.client.xack(stream, group, msg_id)

        return {
            "broker": self.name,
            "pattern": "stream_group",
            "destination": stream,
            "group": group,
            "consumer": consumer,
   

### API 테스트: Stream 상세 정보 (XINFO)

`XINFO STREAM`과 `XINFO GROUPS`로 스트림의 현재 상태를 확인합니다.
각 그룹의 pending(미확인) 메시지 수, 마지막으로 전달된 ID 등을 볼 수 있습니다.

In [24]:
resp = await client.get("/redis/stream/info")
info = resp.json()
print(f"스트림 길이: {info.get('length')}")
print(f"그룹 수: {len(info.get('groups', []))}")
for g in info.get("groups", []):
    print(f"  그룹 '{g['name']}': pending={g['pending']}, consumers={g['consumers']}")

스트림 길이: 6
그룹 수: 1
  그룹 'notebook-group': pending=0, consumers=1


### aioredis 직접 명령: XADD / XRANGE

API 없이 Redis에 직접 Stream 명령을 실행합니다.
`XRANGE`는 ID 범위로 메시지를 조회하는 명령으로, API에는 노출되지 않은 기능입니다.

In [25]:
# 직접 XADD
msg_id = await r.xadd("direct-stream", {"data": "직접 추가", "ts": str(time.time())})
print(f"XADD 결과 ID: {msg_id}")

# XRANGE로 전체 조회 (- 는 최소 ID, + 는 최대 ID)
entries = await r.xrange("direct-stream", "-", "+")
for eid, fields in entries:
    print(f"  {eid}: {fields}")

XADD 결과 ID: 1771384228117-0
  1771384228117-0: {'data': '직접 추가', 'ts': '1771384228.114058'}


### Stream 패턴 요약

| 항목 | 설명 |
|------|------|
| Redis 명령 | `XADD`, `XREAD`, `XREADGROUP`, `XACK`, `XINFO` |
| 메시지 영속성 | O -- 디스크에 저장, ID로 재조회 가능 |
| Consumer Group | O -- 메시지 분배 + PEL 기반 재처리 |
| Kafka와의 차이 | 파티션 없음, 단일 노드 성능 한계, 더 간단한 설정 |
| 적합한 사용처 | 이벤트 소싱, 활동 로그, 경량 스트리밍 |
| API 엔드포인트 | `POST /redis/stream/{stream}/add`, `GET .../read`, `POST .../group` |

---
## 5. Pattern 3: List Queue (LPUSH / BRPOP)

### 은행 번호표 시스템과 같은 원리

Redis List Queue는 **은행 번호표**와 동일한 구조입니다.

```mermaid
  은행에 비유하면:
  ┌─────────────────────────────────────────────────────────┐
  │                                                         │
  │  [고객 입장]                    [창구 직원이 호출]            │
  │                                                         │
  │   고객C ──→ ┌────────────────┐                           │
  │   고객B ──→ │ C │ B │ A      │ ──→ 고객A (1번 창구)         │
  │             └────────────────┘                          │
  │             ↑ 뒤에 추가          ↑ 앞에서 꺼냄               │
  │            (LPUSH)             (BRPOP)                  │
  │                                                         │
  │  규칙:                                                   │
  │  1. 먼저 온 사람이 먼저 처리됨 (FIFO)                         │
  │  2. 한 사람은 하나의 창구에서만 처리됨 (중복 처리 없음)             │
  │  3. 창구가 비면 다음 고객이 올 때까지 대기 (Blocking)            │
  └─────────────────────────────────────────────────────────┘
```

### 왜 List Queue를 사용하나?

실시간 처리가 필요 없는 **무거운 작업**들이 있습니다:
- 이미지 리사이징 (500ms~2s 소요)
- 이메일 발송 (외부 API 호출, 1~3s 소요)
- 보고서 PDF 생성 (5~30s 소요)

이런 작업을 API 요청 중에 처리하면 사용자가 오래 기다려야 합니다.
대신 **큐에 등록하고 바로 응답**을 주면, 백그라운드에서 천천히 처리할 수 있습니다.

```text
  [Without Queue - 사용자가 기다림]
  사용자 → API서버 → 이미지 리사이징(2초) → 응답
                     ↑ 여기서 사용자가 2초 동안 멈춤

  [With Queue - 즉시 응답]
  사용자 → API서버 → 큐에 등록(1ms) → "접수 완료!" 응답
                     ↓
              백그라운드 워커 → 이미지 리사이징(2초) → 완료 알림
```

### BRPOP의 'B'가 중요한 이유

`RPOP`은 큐가 비어있으면 바로 `null`을 반환합니다. 그래서 계속 확인하는 **폴링**이 필요합니다.
`BRPOP`(Blocking RPOP)은 큐가 비어있으면 **새 메시지가 올 때까지 기다립니다**.

```text
  RPOP (Polling 방식):
  워커: "새 작업 있나?" → 없음
  워커: "새 작업 있나?" → 없음     ← CPU와 네트워크 낭비!
  워커: "새 작업 있나?" → 있음! → 처리

  BRPOP (Blocking 방식):
  워커: "새 작업 올 때까지 기다릴게" → (조용히 대기중...)
  (새 작업 도착!) → 즉시 처리      ← 리소스 효율적!
```

### Stream, RabbitMQ와의 차이

| | List Queue | Stream | RabbitMQ |
|---|-----------|--------|----------|
| **ACK/재시도** | X (POP하면 끝) | O (XACK) | O (ACK/NACK) |
| **실패 처리** | 유실됨 | PEL에서 재처리 | DLQ로 이동 |
| **복잡도** | 매우 간단 | 보통 | 높음 |
| **적합한 경우** | 실패해도 괜찮은 간단한 작업 | 실패 시 재처리 필요 | 복잡한 라우팅/보장 필요 |

> **한 줄 요약** — "은행 번호표처럼 먼저 온 작업을 먼저 처리하는 가장 간단한 큐"
> 실패 처리가 중요하지 않은 백그라운드 작업에 적합합니다.

### 소스코드 분석: `queue_push()` / `queue_pop()`

`queue_push()`는 `LPUSH`로 작업을 등록하고, `queue_pop()`는 `BRPOP`으로 가져옵니다.
각 작업에 `job_id`(UUID)와 `queued_at`(타임스탬프)을 자동 부여합니다.

In [26]:
src_push = inspect.getsource(RedisBroker.queue_push)
print(src_push)

    @measure_time(broker="redis_queue", operation="publish")
    async def queue_push(self, queue: str, message: dict) -> dict:
        """List Queue: 작업 등록 (LPUSH)"""
        payload = json.dumps({
            **message,
            "job_id": str(uuid.uuid4())[:8],
            "queued_at": time.time(),
        })
        length = await self.client.lpush(queue, payload)

        return {
            "broker": self.name,
            "pattern": "queue",
            "destination": queue,
            "queue_length": length,
        }



In [27]:
src_pop = inspect.getsource(RedisBroker.queue_pop)
print(src_pop)

    @measure_time(broker="redis_queue", operation="consume", record_metric=False)
    async def queue_pop(self, queue: str, timeout: int = 1) -> dict:
        """List Queue: 작업 가져오기 (BRPOP)"""
        result = await self.client.brpop(queue, timeout=timeout)

        if result:
            _key, value = result
            data = json.loads(value)
            remaining = await self.client.llen(queue)
            return {
                "broker": self.name,
                "pattern": "queue",
                "destination": queue,
                "message": data,
                "remaining": remaining,
            }
        return {
            "broker": self.name,
            "pattern": "queue",
            "destination": queue,
            "message": None,
            "note": f"{timeout}초 대기 후 타임아웃",
        }



### API 테스트: 작업 등록 및 처리

3개의 작업을 큐에 넣고, 하나씩 꺼내봅니다.
FIFO 순서대로 처리되는지 확인합니다.

In [31]:
# 작업 3개 등록
for i in range(3):
    resp = await client.post("/redis/queue/push", json={
        "content": f"작업 #{i+1}",
        "metadata": {"priority": "normal"}
    })
    print(f"PUSH #{i+1}: queue_length={resp.json()['queue_length']}")

PUSH #1: queue_length=1
PUSH #2: queue_length=2
PUSH #3: queue_length=3


In [ ]:
# 큐 길이 확인 후 하나씩 꺼내기
resp = await client.get("/redis/queue/length")
print(f"현재 큐 길이: {resp.json()['length']}\n")

for i in range(3):
    resp = await client.get("/redis/queue/pop")
    data = resp.json()
    msg = data.get("message", {})
    print(f"POP #{i+1}: {msg.get('content', 'empty')}, remaining={data.get('remaining')}")

현재 큐 길이: 3

POP #1: 작업 #1, remaining=2
POP #2: 작업 #2, remaining=1
POP #3: 작업 #3, remaining=0


### aioredis 직접 명령: LPUSH / LRANGE / RPOP

`LRANGE`를 사용하면 큐에서 꺼내지 않고도 내용을 확인할 수 있습니다.
디버깅이나 모니터링에 유용합니다.

In [34]:
# 직접 LPUSH
await r.lpush("direct-queue", "task-C", "task-B", "task-A")

# LRANGE로 전체 확인 (꺼내지 않음)
items = await r.lrange("direct-queue", 0, -1)
print(f"큐 내용 (LRANGE): {items}")

# RPOP으로 하나 꺼내기
item = await r.rpop("direct-queue")
print(f"RPOP 결과: {item} (FIFO: 가장 먼저 넣은 것이 나옴)")

큐 내용 (LRANGE): ['task-A', 'task-B', 'task-C', 'task-A', 'task-B']
RPOP 결과: task-B (FIFO: 가장 먼저 넣은 것이 나옴)


### List Queue 패턴 요약

| 항목 | 설명 |
|------|------|
| Redis 명령 | `LPUSH`, `BRPOP`, `LRANGE`, `LLEN` |
| 처리 보장 | 한 번만 소비 (POP하면 리스트에서 제거) |
| Blocking 지원 | O -- `BRPOP`이 새 메시지까지 대기 |
| RabbitMQ와의 차이 | ACK/NACK 없음, DLQ 없음, 라우팅 없음 |
| 적합한 사용처 | 백그라운드 작업, 간단한 태스크 큐 |
| API 엔드포인트 | `POST /redis/queue/{q}/push`, `POST .../pop`, `GET .../length` |

---
## 6. Pattern 4: Cache (SET/GET + TTL)

### 포스트잇 메모와 같은 원리

캐시는 **자주 찾는 정보를 포스트잇에 적어 책상 위에 붙여놓는 것**과 같습니다.

```text
  [캐시 없이 매번 DB 조회]
  
  사용자: "홍길동 정보 알려줘"
  서버: DB야, 홍길동 찾아줘 → (디스크 검색... 50ms) → 결과 반환
  
  사용자: "홍길동 정보 다시 알려줘"
  서버: DB야, 또 홍길동 찾아줘 → (또 디스크 검색... 50ms) → 결과 반환
  
  ↑ 같은 정보를 매번 느리게 찾고 있음!
  
  ─────────────────────────────────────────────
  
  [캐시 사용]
  
  사용자: "홍길동 정보 알려줘"
  서버: Redis에 있나? → 없음 (Cache Miss)
        → DB에서 찾기 (50ms) → 결과를 Redis에 저장 → 반환
  
  사용자: "홍길동 정보 다시 알려줘"  
  서버: Redis에 있나? → 있음! (Cache Hit, 0.1ms)
        → 바로 반환!  ← 500배 빨라짐!
```

### TTL(Time-To-Live)이 반드시 필요한 이유

포스트잇에 적힌 정보가 영원히 유효할까요? 아닙니다.
- 홍길동이 이메일을 변경했다면?
- 상품 가격이 할인으로 바뀌었다면?

캐시에 저장된 정보가 원본(DB)과 달라지는 것을 **캐시 불일치**라고 합니다.
이를 방지하기 위해 **TTL(만료 시간)**을 설정합니다.

```text
  SET "user:1001" "{이름:홍길동}" EX 60
                                   ↑
                                60초 후 자동 삭제!

  시간 흐름:
  [0초]  저장됨 → GET → Hit! (데이터 반환)
  [30초] GET → Hit! (TTL 30초 남음)
  [60초] 자동 삭제됨
  [61초] GET → Miss (다시 DB에서 가져와야 함)
```

### Cache Hit Rate: 캐시가 잘 동작하는지 확인하는 지표

```text
  Cache Hit Rate = Cache Hit 수 / 전체 요청 수 × 100

  예시:
  - 100번 요청 중 80번이 캐시에서 응답 → Hit Rate 80% (양호)
  - 100번 요청 중 20번이 캐시에서 응답 → Hit Rate 20% (비효율)
  
  Hit Rate가 낮다면:
  - TTL이 너무 짧은 건 아닌지?
  - 캐시할 데이터를 잘못 선택한 건 아닌지?
  - 메모리가 부족해서 캐시가 밀려나는 건 아닌지?
```

### 캐시 전략: Cache-Aside 패턴

이 프로젝트에서 사용하는 캐시 전략은 **Cache-Aside** (또는 Lazy Loading)입니다.
애플리케이션이 직접 캐시를 관리합니다.

```text
  [읽기 흐름]                    [쓰기 흐름]
  
  1. 캐시에서 찾기               1. DB에 저장
     ├── Hit → 반환              2. 캐시 삭제 (무효화)
     └── Miss                      → 다음 읽기 때 새 데이터가 캐시됨
         → 2. DB에서 읽기
         → 3. 캐시에 저장 (TTL)
         → 4. 반환
```

> **한 줄 요약** — "자주 찾는 정보를 포스트잇에 적어두되, 일정 시간 후에는 버리고 새로 적는 것"
> DB 부하를 줄이고 응답 속도를 높이는 가장 기본적인 최적화 기법입니다.

### 소스코드 분석: `cache_set()` / `cache_get()`

`cache_get()`은 단순히 값을 반환하는 것이 아니라, `hit` 여부와 `remaining_ttl`도
함께 반환합니다. 이를 통해 캐시 효율을 모니터링할 수 있습니다.

In [35]:
src_set = inspect.getsource(RedisBroker.cache_set)
print(src_set)

    @measure_time(broker="redis_cache", operation="publish")
    async def cache_set(self, key: str, value: dict, ttl: int = 60) -> dict:
        """Cache: 값 저장 (TTL 기본 60초)"""
        payload = json.dumps(value)
        await self.client.set(key, payload, ex=ttl)

        return {
            "broker": self.name,
            "pattern": "cache",
            "operation": "SET",
            "key": key,
            "ttl_seconds": ttl,
        }



In [36]:
src_get = inspect.getsource(RedisBroker.cache_get)
print(src_get)

    @measure_time(broker="redis_cache", operation="consume", record_metric=False)
    async def cache_get(self, key: str) -> dict:
        """Cache: 값 조회 (hit/miss 반환)"""
        raw = await self.client.get(key)

        if raw:
            ttl = await self.client.ttl(key)
            return {
                "broker": self.name,
                "pattern": "cache",
                "operation": "GET",
                "key": key,
                "hit": True,
                "value": json.loads(raw),
                "remaining_ttl": ttl,
            }
        return {
            "broker": self.name,
            "pattern": "cache",
            "operation": "GET",
            "key": key,
            "hit": False,
            "value": None,
        }



### API 테스트: Cache Hit / Miss 시나리오

1. 캐시에 값을 저장 (TTL=30초)
2. 즉시 조회 -> Cache Hit
3. 삭제 후 조회 -> Cache Miss

이 패턴은 실무에서 "Cache-Aside" 전략의 기본 흐름입니다.

In [37]:
# 1. 캐시 저장
resp = await client.post("/redis/cache/set", json={
    "key": "user:1001",
    "value": {"name": "홍길동", "email": "hong@example.com"},
    "ttl": 30
})
print("SET:", resp.json())

SET: {'broker': 'redis', 'pattern': 'cache', 'operation': 'SET', 'key': 'user:1001', 'ttl_seconds': 30, 'elapsed_ms': 52.535}


In [41]:
# 2. Cache Hit 확인
resp = await client.get("/redis/cache/get/user:1001")
result = resp.json()
print(f"Hit: {result['hit']}, TTL 남음: {result['remaining_ttl']}초")
print(f"Value: {result['value']}")

Hit: True, TTL 남음: 23초
Value: {'name': '홍길동', 'email': 'hong@example.com'}


In [42]:
# 3. 삭제 후 Cache Miss
await client.delete("/redis/cache/delete/user:1001")
resp = await client.get("/redis/cache/get/user:1001")
result = resp.json()
print(f"Hit: {result['hit']}, Value: {result['value']}")
print("-> 삭제 후에는 miss 반환")

Hit: False, Value: None
-> 삭제 후에는 miss 반환


### aioredis 직접 명령: SET / GET / TTL

직접 명령으로 TTL 동작을 더 세밀하게 관찰합니다.
`TTL` 명령은 키가 없으면 -2, TTL이 없으면 -1, 있으면 남은 초를 반환합니다.

In [46]:
await r.set("demo:ttl", "expire-test", ex=10)
val = await r.get("demo:ttl")
ttl = await r.ttl("demo:ttl")
print(f"GET: {val}, TTL: {ttl}초 남음")

# 존재하지 않는 키의 TTL
ttl_none = await r.ttl("nonexistent-key")
print(f"존재하지 않는 키 TTL: {ttl_none} (-2 = 키 없음)")

GET: expire-test, TTL: 10초 남음
존재하지 않는 키 TTL: -2 (-2 = 키 없음)


### Cache 패턴 요약

| 항목 | 설명 |
|------|------|
| Redis 명령 | `SET key value EX ttl`, `GET key`, `DEL key`, `TTL key` |
| 자동 만료 | O -- TTL 만료 시 Redis가 자동 삭제 |
| 캐시 전략 | Cache-Aside (Read-Through도 가능) |
| 주의사항 | Cache Stampede, 캐시 무효화 타이밍 |
| 적합한 사용처 | DB 조회 결과 캐싱, 세션 저장, API 응답 캐싱 |
| API 엔드포인트 | `POST /redis/cache`, `GET /redis/cache/{key}`, `DELETE /redis/cache/{key}` |

---
## 7. Pattern 5: Rate Limiter (슬라이딩 윈도우)

### 놀이공원 입장 제한과 같은 원리

인기 놀이기구에는 **시간당 탑승 인원 제한**이 있습니다. Rate Limiter도 같은 원리입니다.

```text
  놀이공원 입장 제한에 비유:
  ┌───────────────────────────────────────────────────┐
  │                                                   │
  │  규칙: "1시간에 최대 100명만 입장 가능"             │
  │                                                   │
  │  14:00~15:00                                      │
  │  ├── 97번째 손님 → "입장하세요" (remaining: 3)     │
  │  ├── 98번째 손님 → "입장하세요" (remaining: 2)     │
  │  ├── 99번째 손님 → "입장하세요" (remaining: 1)     │
  │  ├── 100번째 손님 → "입장하세요" (remaining: 0)    │
  │  └── 101번째 손님 → "죄송합니다, 잠시 후에 오세요"  │
  │                                                   │
  │  15:01 → 14:01 이전 손님 기록 자동 삭제 → 다시 입장 가능!
  └───────────────────────────────────────────────────┘
```

### 왜 Rate Limiter가 필요한가?

API 서버를 운영하면 반드시 마주치는 문제들입니다:

1. **악의적 남용**: 한 사용자가 초당 1000번 API를 호출 → 서버 다운
2. **비용 폭탄**: 외부 API(문자 발송, AI 호출)를 무제한 호출 → 청구서 폭탄
3. **공정한 사용**: 한 사용자가 모든 리소스를 독점 → 다른 사용자 서비스 불가

```text
  Rate Limiter 없이:
  사용자A → [요청 1000개/초] → 서버 과부하 → 전체 사용자 서비스 중단
  
  Rate Limiter 있으면:
  사용자A → [요청 10개/초 초과] → "429 Too Many Requests" 반환
         → 나머지 사용자는 정상 서비스
```

### 고정 윈도우 vs 슬라이딩 윈도우

이 프로젝트에서 **슬라이딩 윈도우** 방식을 사용하는 이유가 있습니다.

```text
  [고정 윈도우의 문제 — 경계에서 2배 요청 가능]
  
  규칙: 1분에 10개 제한
  
  시간: ...━━━━━━━┃━━━━━━━━┃━━━━━━━━...
         00:00분  00:01분   00:02분
  
  00:00:55에 10개 요청 (OK)
  00:01:05에 10개 요청 (OK ← 새 윈도우니까!)
  → 10초 사이에 20개 요청이 통과됨!
  
  ─────────────────────────────────────
  
  [슬라이딩 윈도우 — 항상 정확]
  
  현재 시각이 00:01:05라면,
  "00:00:05 ~ 00:01:05" 범위의 요청만 센다
  
  → 어느 시점에서 보더라도 1분 내 요청이 10개를 넘지 않음!
```

### Redis Sorted Set으로 구현하는 원리

슬라이딩 윈도우를 구현하기 위해 **Sorted Set**을 사용합니다.
각 요청의 타임스탬프를 score로 저장하면, 시간 범위 검색이 가능합니다.

```text
  Sorted Set: ratelimit:user-123
  ┌──────────────────────────────────────┐
  │  member(요청ID)    score(시간)        │
  │  "req-abc"         1700000050.123    │
  │  "req-def"         1700000051.456    │
  │  "req-ghi"         1700000055.789    │ ← 이 범위만 세면 됨
  │  "req-jkl"         1700000058.012    │
  └──────────────────────────────────────┘
  
  Pipeline으로 4단계를 한 번에 실행:
  1. ZREMRANGEBYSCORE → 윈도우 밖 오래된 기록 삭제
  2. ZCARD            → 현재 윈도우 내 요청 수 확인
  3. ZADD             → 이번 요청 기록 추가
  4. EXPIRE           → 키 자동 만료 설정 (메모리 누수 방지)
```

### Pipeline이 여기서 중요한 이유

위 4개 명령을 **개별로** 실행하면 위험합니다:
- 1번(삭제)과 2번(카운트) 사이에 다른 요청이 끼어들 수 있음
- 2번(카운트)과 3번(추가) 사이에 카운트가 바뀔 수 있음

Pipeline은 4개 명령을 **네트워크 1회 왕복**으로 묶어 보내므로,
다른 요청이 중간에 끼어들 가능성을 크게 줄입니다.

> **한 줄 요약** — "놀이공원처럼 시간대별 방문자 수를 추적하여, 제한을 초과하면 입장을 막는 것"
> Sorted Set의 score를 타임스탬프로 활용하는 것이 핵심 아이디어입니다.

### 소스코드 분석: `rate_limit_check()`

이 메서드는 4개의 Redis 명령을 **Pipeline**으로 원자적으로 실행합니다:
1. `ZREMRANGEBYSCORE` -- 윈도우 밖의 오래된 기록 삭제
2. `ZCARD` -- 현재 윈도우 내 요청 수 확인
3. `ZADD` -- 현재 요청을 기록
4. `EXPIRE` -- 키 자동 만료 설정

Pipeline을 사용하는 이유: 4개 명령을 네트워크 왕복 1회로 처리하여 레이스 컨디션을 방지합니다.

In [47]:
src = inspect.getsource(RedisBroker.rate_limit_check)
print(src)

    async def rate_limit_check(
        self, key: str, max_requests: int = 10, window_seconds: int = 60
    ) -> dict:
        """Rate Limiter: 요청 가능 여부 확인 + 기록"""
        now = time.time()
        window_start = now - window_seconds
        redis_key = f"ratelimit:{key}"

        pipe = self.client.pipeline()
        pipe.zremrangebyscore(redis_key, 0, window_start)
        pipe.zcard(redis_key)
        pipe.zadd(redis_key, {f"{now}:{uuid.uuid4().hex[:6]}": now})
        pipe.expire(redis_key, window_seconds)
        results = await pipe.execute()

        current_count = results[1]
        allowed = current_count < max_requests

        return {
            "broker": self.name,
            "pattern": "rate_limiter",
            "key": key,
            "allowed": allowed,
            "current_requests": current_count,
            "max_requests": max_requests,
            "window_seconds": window_seconds,
            "remaining": max(0, max_requests - current_count),
        }



### API 테스트: Rate Limiter 동작 확인

5회 제한(10초 윈도우)으로 설정하고 연속 호출하여
`allowed`가 `False`로 바뀌는 시점을 확인합니다.

In [54]:
for i in range(7):
    resp = await client.get("/redis/ratelimit/check", params={
        "key": "notebook-test",
        "max_requests": 5,
        "window_seconds": 10
    })
    result = resp.json()
    status = "PASS" if result["allowed"] else "BLOCKED"
    print(f"요청 #{i+1}: {status} (count={result['current_requests']}, remaining={result['remaining']})")

요청 #1: PASS (count=0, remaining=5)
요청 #2: PASS (count=1, remaining=4)
요청 #3: PASS (count=2, remaining=3)
요청 #4: PASS (count=3, remaining=2)
요청 #5: PASS (count=4, remaining=1)
요청 #6: BLOCKED (count=5, remaining=0)
요청 #7: BLOCKED (count=6, remaining=0)


### aioredis 직접 명령: Sorted Set 내부 확인

Rate Limiter의 내부를 직접 들여다봅니다.
`ZRANGEBYSCORE`로 Sorted Set에 저장된 요청 기록을 확인할 수 있습니다.

In [55]:
# Rate Limiter의 Sorted Set 내부 확인
entries = await r.zrange("ratelimit:notebook-test", 0, -1, withscores=True)
print(f"Sorted Set 항목 수: {len(entries)}")
for member, score in entries[:5]:
    print(f"  member={member[:30]}..., score(timestamp)={score}")

Sorted Set 항목 수: 7
  member=1771500690.3870149:cc730e..., score(timestamp)=1771500690.3870149
  member=1771500690.4006479:a7c1bc..., score(timestamp)=1771500690.4006479
  member=1771500690.41423:0ed8c5..., score(timestamp)=1771500690.41423
  member=1771500690.435172:4c5c8f..., score(timestamp)=1771500690.435172
  member=1771500690.437056:be47dc..., score(timestamp)=1771500690.437056


### Pipeline 직접 실습

Pipeline은 여러 명령을 묶어 한 번에 실행합니다.
개별 호출 대비 네트워크 RTT를 크게 줄일 수 있습니다.

In [56]:
# Pipeline vs 개별 호출 비교
import asyncio

# Pipeline: 1회 왕복
t0 = time.time()
pipe = r.pipeline()
for i in range(100):
    pipe.set(f"bench:{i}", f"value-{i}")
await pipe.execute()
pipeline_ms = (time.time() - t0) * 1000

# 개별 호출: 100회 왕복
t0 = time.time()
for i in range(100):
    await r.set(f"bench2:{i}", f"value-{i}")
individual_ms = (time.time() - t0) * 1000

print(f"Pipeline (100건): {pipeline_ms:.1f}ms")
print(f"개별 호출 (100건): {individual_ms:.1f}ms")
print(f"Pipeline이 {individual_ms/pipeline_ms:.1f}배 빠름")

Pipeline (100건): 22.9ms
개별 호출 (100건): 15.2ms
Pipeline이 0.7배 빠름


### Rate Limiter 패턴 요약

| 항목 | 설명 |
|------|------|
| Redis 명령 | `ZADD`, `ZREMRANGEBYSCORE`, `ZCARD`, `EXPIRE` (Pipeline) |
| 윈도우 방식 | 슬라이딩 윈도우 -- 경계 문제 없음 |
| 원자성 | Pipeline으로 4개 명령 원자적 실행 |
| 키 형식 | `ratelimit:{key}` (자동 만료) |
| 적합한 사용처 | API 속도 제한, 로그인 시도 제한, DDoS 방어 |
| API 엔드포인트 | `GET /redis/ratelimit/check` |

---
## 11-1. 보너스: 분산 Lock (SET NX)

Redis를 사용한 **분산 Lock**은 여러 프로세스/서버가 동시에 같은 리소스에 접근하는 것을 방지합니다.
`SET key value NX EX ttl` 명령 하나로 원자적 lock 획득이 가능합니다.

### 왜 필요한가?
- 재고 차감: 2개 서버가 동시에 "재고 1개 남음"을 읽고 각각 차감하면 **-1개**가 됨
- 결제 처리: 같은 주문을 2번 결제하는 **중복 결제** 방지
- 배치 작업: 크론잡이 2대 서버에서 동시 실행되는 것을 방지

### 핵심 원리
```text
SET lock:order-123 "worker-A" NX EX 10
  NX = 키가 없을 때만 설정 (Not eXists)
  EX = 10초 후 자동 만료 (데드락 방지)
```

In [ ]:
---
## 11. aioredis 직접 실습 - 관리 명령어

Redis 서버의 전체 상태를 파악하는 관리용 명령어들입니다.
운영 환경에서 디버깅이나 모니터링에 자주 사용됩니다.

In [ ]:
bf_key = f"crawler:{int(time.time())}"
print(f"=== Bloom Filter: {bf_key} ===")

# 1. URL 100개 추가
for i in range(100):
    await client.post("/redis/bloom/add", json={
        "filter_key": bf_key,
        "item": f"https://site.com/page/{i}"
    })
print("100개 URL 추가 완료\n")

# 2. 이미 추가한 URL → exists True
r1 = await client.post("/redis/bloom/exists", json={
    "filter_key": bf_key, "item": "https://site.com/page/42"
})
print(f"존재하는 URL (page/42): {r1.json()}")

# 3. 추가하지 않은 URL → exists False
r2 = await client.post("/redis/bloom/exists", json={
    "filter_key": bf_key, "item": "https://site.com/page/999"
})
print(f"없는 URL (page/999): {r2.json()}")

# 4. Bloom Filter 정보 조회
r3 = await client.get(f"/redis/bloom/info/{bf_key}")
print(f"\nBloom Filter 정보: {r3.json()}")

### Bloom Filter 패턴 요약

| 항목 | 설명 |
|------|------|
| 구현 | Redis BITFIELD + double hashing (Kirsch-Mitzenmacher) |
| False Negative | **절대 없음** — 있는 것을 없다고 하지 않음 |
| False Positive | 낮은 확률 (기본 설정: ~1%) |
| 메모리 절감 | Set 대비 83배 이상 절감 |
| 적합한 사용처 | 크롤러 중복 URL, 스팸 필터, 회원가입 이메일 중복 체크 |
| API 엔드포인트 | `POST /redis/bloom/add`, `POST /redis/bloom/exists`, `GET /redis/bloom/info/{key}` |

---
## 9. Pattern 7: TimeSeries — 시간 기반 메트릭 저장소

### 핵심 아이디어: Sorted Set의 score = 밀리초 타임스탬프

Redis TimeSeries는 별도 모듈 없이 **Sorted Set**으로 구현합니다.
score를 타임스탬프(ms)로 사용하면 시간 범위 쿼리가 자연스럽게 가능합니다.

```text
  Sorted Set: ts:cpu:host1
  ┌──────────────────────────────────────────┐
  │  member(직렬화 값)       score(ms 타임스탬프) │
  │  "72.5"                 1700000001000    │  ← 14:00:01
  │  "68.3"                 1700000060000    │  ← 14:01:00
  │  "75.1"                 1700000120000    │  ← 14:02:00
  │  "91.2"                 1700000180000    │  ← 14:03:00
  └──────────────────────────────────────────┘

  ZRANGEBYSCORE ts:cpu:host1 start_ms end_ms → 시간 범위 조회
```

### 4가지 핵심 오퍼레이션

| 오퍼레이션 | Redis 명령 | 설명 |
|-----------|-----------|------|
| **ts_add** | ZADD score=timestamp_ms | 타임스탬프 + 값 추가 |
| **ts_range** | ZRANGEBYSCORE | 시간 범위 조회 |
| **ts_latest** | ZRANGE (rev, limit N) | 최신 N개 조회 |
| **ts_trim** | ZREMRANGEBYSCORE | 오래된 데이터 삭제 (retention policy) |

> **beanllm 활용** — AI 모델 호출 레이턴시를 초당 측정하여
> 이상 급증(spike) 감지 및 SLA 모니터링에 활용합니다.

In [ ]:
ts_key = f"cpu:host1:{int(time.time())}"
cpu_values = [72.5, 68.3, 75.1, 91.2, 88.7, 95.3, 60.1, 55.8, 58.2, 62.4]

print(f"=== TimeSeries: {ts_key} ===")

# 1. 10개 메트릭 포인트 추가 (1초 간격 시뮬레이션)
base_ms = int(time.time() * 1000) - len(cpu_values) * 1000
for i, val in enumerate(cpu_values):
    ts_ms = base_ms + i * 1000
    await client.post("/redis/ts/add", json={
        "series_key": ts_key, "value": val, "timestamp_ms": ts_ms
    })
print(f"{len(cpu_values)}개 포인트 추가 완료\n")

# 2. 전체 범위 조회
resp = await client.get(f"/redis/ts/range/{ts_key}", params={
    "start_ms": 0, "end_ms": 9999999999999
})
data = resp.json()
print(f"전체 포인트 수: {len(data['points'])}")
for pt in data["points"]:
    print(f"  ts={pt['timestamp_ms']} → cpu={pt['value']}%")

# 3. 최신 3개 조회
resp = await client.get(f"/redis/ts/latest/{ts_key}", params={"count": 3})
print(f"\n최신 3개: {[p['value'] for p in resp.json()['points']]}")

# 4. 오래된 데이터 삭제 (5초 이전)
cutoff_ms = base_ms + 5 * 1000
resp = await client.delete(f"/redis/ts/trim/{ts_key}", params={"before_ms": cutoff_ms})
print(f"\n5초 이전 데이터 삭제 결과: {resp.json()}")

### TimeSeries 패턴 요약

| 항목 | 설명 |
|------|------|
| 구현 | Redis Sorted Set (score = timestamp_ms) |
| 저장 단위 | 타임스탬프(ms) + 값 |
| 시간 범위 쿼리 | ZRANGEBYSCORE (O(log N + M)) |
| Retention Policy | ZREMRANGEBYSCORE로 오래된 데이터 자동 삭제 |
| 적합한 사용처 | 서버 메트릭, AI 레이턴시 추적, IoT 센서 데이터 |
| API 엔드포인트 | `POST /redis/ts/add`, `GET /redis/ts/range/{key}`, `GET /redis/ts/latest/{key}`, `DELETE /redis/ts/trim/{key}` |

---
## 10. Pattern 8: Vector Set — AI 시맨틱 검색 (Redis 8.0+)

### 핵심 아이디어: 단어가 아닌 "의미"로 검색한다

```text
  일반 키워드 검색:                    벡터 검색 (시맨틱 검색):
  "강아지" 검색 →                     [0.95, 0.31, 0.0, 0.0] 검색 →
  "강아지"가 포함된 문서만              의미가 비슷한 문서를 코사인 유사도로 순위 반환

  "반려견 케어" 문서 → 검색 안 됨!    "반려견-케어-가이드" → cosine sim=0.98  ✅
                                     "강아지에-대한-문서" → cosine sim=0.95  ✅
                                     "자동차-정비-방법"   → cosine sim=0.00  ❌
```

### Redis 8.0 VADD / VSIM 명령어

```text
  VADD vset_key REDUCE 4 VALUES 1.0 0.0 0.0 0.0 element-id
   ↑    ↑              ↑       ↑_________________↑
  명령  키 이름    차원 수         벡터 값 (4차원)  요소 ID

  VSIM vset_key VALUES 4 0.95 0.31 0.0 0.0 COUNT 3
  → 코사인 유사도 상위 3개 반환
```

### 코사인 유사도(Cosine Similarity) 이해

```text
  두 벡터의 방향이 같을수록 유사도 1.0에 가까워짐

  [1.0, 0.0] →    와    [0.9, 0.44] → 의 각도가 작음 → sim ≈ 0.99
  [1.0, 0.0] →    와    [0.0, 1.0]  ↑ 의 각도가 90도 → sim = 0.00
```

> **Redis 8.0 전용** — `VADD`, `VSIM` 명령어는 Redis 8.0 이상에서만 사용 가능합니다.
> 이 프로젝트는 Redis 8.4를 사용하므로 바로 실습 가능합니다.

> **beanllm 활용** — 사용자 쿼리를 임베딩 벡터로 변환하여 Redis Vector Set에서
> 유사한 문서를 검색하는 RAG 파이프라인에 활용합니다.

In [ ]:
vset_key = f"docs:{int(time.time())}"

docs = [
    ("강아지에-대한-문서",  [1.0, 0.0, 0.0, 0.0]),
    ("반려견-케어-가이드",  [0.9, 0.44, 0.0, 0.0]),
    ("자동차-정비-방법",   [0.0, 1.0, 0.0, 0.0]),
    ("AI-머신러닝-소개",   [0.0, 0.0, 1.0, 0.0]),
]

print(f"=== Vector Set: {vset_key} ===")

# 1. 문서 벡터 추가
for elem_id, vec in docs:
    resp = await client.post("/redis/vector/add", json={
        "vset_key": vset_key,
        "element_id": elem_id,
        "vector": vec
    })
    if resp.status_code != 200:
        print(f"  VADD 실패 ({resp.status_code}): {resp.text[:200]}")
        break
print(f"{len(docs)}개 문서 벡터 추가 완료\n")

# 2. 유사도 검색: "강아지 관련 문서 찾기"
query_vector = [0.95, 0.31, 0.0, 0.0]
resp = await client.post("/redis/vector/search", json={
    "vset_key": vset_key,
    "query_vector": query_vector,
    "top_k": 4
})
result = resp.json()
print(f"쿼리 벡터: {query_vector}")
print(f"유사도 상위 {len(result['results'])}개 결과:")
for item in result["results"]:
    print(f"  {item['element_id']}: score={item['score']:.4f}")

---
## 12. 정리 및 요약 테이블

### 8가지 패턴 비교

| 패턴 | 데이터 구조 | 영속성 | 소비 방식 | 대표 명령 | 사용처 |
|------|-----------|--------|----------|----------|-------|
| Pub/Sub | Channel | X | 브로드캐스트 | PUBLISH/SUBSCRIBE | 실시간 알림 |
| Stream | Stream | O | Consumer Group | XADD/XREADGROUP | 이벤트 로그 |
| List Queue | List | O | POP (1:1) | LPUSH/BRPOP | 작업 큐 |
| Cache | String | O (TTL) | GET/SET | SET EX/GET | 응답 캐싱 |
| Rate Limiter | Sorted Set | O | Pipeline 조회 | ZADD/ZCARD | API 제한 |
| **Bloom Filter** | Bit Array | O | BITFIELD 조회 | BITFIELD/BITPOS | 중복 필터링 |
| **TimeSeries** | Sorted Set | O | ZRANGEBYSCORE | ZADD/ZRANGEBYSCORE | 메트릭 저장 |
| **Vector Set** | Vector Set | O | VSIM 검색 | VADD/VSIM | 시맨틱 검색 |

### RedisBroker 메서드 매핑

| 메서드 | Redis 명령 | API 엔드포인트 |
|--------|-----------|---------------|
| `publish()` | PUBLISH | POST `/redis/pubsub/publish` |
| `subscribe()` | SUBSCRIBE | (서버 내부 사용) |
| `stream_add()` | XADD | POST `/redis/stream/add` |
| `stream_read()` | XREAD | GET `/redis/stream/read` |
| `stream_create_group()` | XGROUP CREATE | POST `/redis/stream/group/create` |
| `stream_read_group()` | XREADGROUP + XACK | GET `/redis/stream/group/read` |
| `stream_info()` | XINFO | GET `/redis/stream/info` |
| `queue_push()` | LPUSH | POST `/redis/queue/push` |
| `queue_pop()` | BRPOP | GET `/redis/queue/pop` |
| `queue_length()` | LLEN | GET `/redis/queue/length` |
| `cache_set()` | SET EX | POST `/redis/cache/set` |
| `cache_get()` | GET + TTL | GET `/redis/cache/get/{key}` |
| `cache_delete()` | DEL | DELETE `/redis/cache/delete/{key}` |
| `rate_limit_check()` | Pipeline(ZREMRANGEBYSCORE+ZCARD+ZADD+EXPIRE) | GET `/redis/ratelimit/check` |
| `bloom_add()` | BITFIELD (double hash) | POST `/redis/bloom/add` |
| `bloom_exists()` | BITFIELD GET | POST `/redis/bloom/exists` |
| `bloom_info()` | BITCOUNT + DEBUG | GET `/redis/bloom/info/{key}` |
| `ts_add()` | ZADD score=ts_ms | POST `/redis/ts/add` |
| `ts_range()` | ZRANGEBYSCORE | GET `/redis/ts/range/{key}` |
| `ts_latest()` | ZRANGE (rev, limit) | GET `/redis/ts/latest/{key}` |
| `ts_trim()` | ZREMRANGEBYSCORE | DELETE `/redis/ts/trim/{key}` |
| `vector_add()` | VADD | POST `/redis/vector/add` |
| `vector_search()` | VSIM | POST `/redis/vector/search` |

### 핵심 교훈

1. **Redis는 만능이 아닙니다** -- 각 패턴의 한계를 이해하고 적합한 곳에 사용
2. **Pub/Sub vs Stream** -- 유실 가능 여부가 선택의 핵심 기준
3. **Pipeline** -- 여러 명령을 원자적으로 실행할 때 반드시 사용
4. **TTL** -- 캐시는 반드시 만료 시간을 설정해야 메모리 누수 방지
5. **Redis 8 고급 패턴** -- Bloom Filter(메모리 83배 절감), TimeSeries(Sorted Set 활용), Vector Set(Redis 8.0+ AI 검색)

---
## 8-1. 보너스: 분산 Lock (SET NX)

Redis를 사용한 **분산 Lock**은 여러 프로세스/서버가 동시에 같은 리소스에 접근하는 것을 방지합니다.
`SET key value NX EX ttl` 명령 하나로 원자적 lock 획득이 가능합니다.

### 왜 필요한가?
- 재고 차감: 2개 서버가 동시에 "재고 1개 남음"을 읽고 각각 차감하면 **-1개**가 됨
- 결제 처리: 같은 주문을 2번 결제하는 **중복 결제** 방지
- 배치 작업: 크론잡이 2대 서버에서 동시 실행되는 것을 방지

### 핵심 원리
```text
SET lock:order-123 "worker-A" NX EX 10
  NX = 키가 없을 때만 설정 (Not eXists)
  EX = 10초 후 자동 만료 (데드락 방지)
```


# 정리: 연결 종료
await r.aclose()
await client.aclose()
print("노트북 02 완료 -- Redis 8가지 패턴 학습 끝")

### KEYS / TYPE: 키 탐색

`KEYS *`는 모든 키를 반환합니다. 프로덕션에서는 `SCAN`을 사용해야 하지만,
학습 환경에서는 `KEYS`로 전체 상태를 빠르게 파악할 수 있습니다.

In [66]:
keys = await r.keys("*")
scans = await r.scan()
print(f"전체 키 수: {len(keys)}\n")
print(f"전체 키 수 with scan: {scans}\n")

for key in sorted(keys)[:15]:
    key_type = await r.type(key)
    print(f"  {key_type:8s} | {key}")

전체 키 수: 203

전체 키 수 with scan: (8, ['bench2:48', 'bench:90', 'bench2:72', 'bench:44', 'bench2:67', 'bench:89', 'bench:53', 'bench:19', 'bench2:1', 'bench2:14'])

  string   | bench2:0
  string   | bench2:1
  string   | bench2:10
  string   | bench2:11
  string   | bench2:12
  string   | bench2:13
  string   | bench2:14
  string   | bench2:15
  string   | bench2:16
  string   | bench2:17
  string   | bench2:18
  string   | bench2:19
  string   | bench2:2
  string   | bench2:20
  string   | bench2:21


### INFO: 서버 상태 확인

`INFO` 명령은 Redis 서버의 상세 통계를 반환합니다.
메모리 사용량, 연결 수, 초당 명령 처리량 등을 확인할 수 있습니다.

In [67]:
info = await r.info("server")
print(f"Redis 버전: {info['redis_version']}")
print(f"실행 모드: {info.get('redis_mode', 'standalone')}")
print(f"프로세스 ID: {info['process_id']}")
print(f"TCP 포트: {info['tcp_port']}")

Redis 버전: 8.4.0
실행 모드: standalone
프로세스 ID: 1091
TCP 포트: 6379


In [68]:
mem = await r.info("memory")
print(f"사용 메모리: {mem['used_memory_human']}")
print(f"최대 메모리: {mem.get('maxmemory_human', 'unlimited')}")
print(f"메모리 정책: {mem.get('maxmemory_policy', 'noeviction')}")

stats = await r.info("stats")
print(f"\n총 처리 명령 수: {stats['total_commands_processed']}")
print(f"총 연결 수: {stats['total_connections_received']}")

사용 메모리: 1.72M
최대 메모리: 0B
메모리 정책: noeviction

총 처리 명령 수: 96243
총 연결 수: 72


### DBSIZE / FLUSHDB: 데이터베이스 관리

`DBSIZE`는 현재 DB의 키 수를 반환합니다.
`FLUSHDB`는 모든 키를 삭제하므로 **프로덕션에서는 절대 사용하지 마세요.**

In [69]:
db_size = await r.dbsize()
print(f"현재 DB 키 수: {db_size}")

# 테스트용 키 정리 (bench:*, bench2:*, direct-*, demo:* 만 삭제)
cleanup_patterns = ["bench:*", "bench2:*", "direct-*", "demo:*"]
deleted = 0
for pattern in cleanup_patterns:
    for key in await r.keys(pattern):
        await r.delete(key)
        deleted += 1
print(f"정리된 테스트 키: {deleted}개")

현재 DB 키 수: 203
정리된 테스트 키: 202개


---
## 9. 정리 및 요약 테이블

### 5가지 패턴 비교

| 패턴 | 데이터 구조 | 영속성 | 소비 방식 | 대표 명령 | 사용처 |
|------|-----------|--------|----------|----------|-------|
| Pub/Sub | Channel | X | 브로드캐스트 | PUBLISH/SUBSCRIBE | 실시간 알림 |
| Stream | Stream | O | Consumer Group | XADD/XREADGROUP | 이벤트 로그 |
| List Queue | List | O | POP (1:1) | LPUSH/BRPOP | 작업 큐 |
| Cache | String | O (TTL) | GET/SET | SET EX/GET | 응답 캐싱 |
| Rate Limiter | Sorted Set | O | Pipeline 조회 | ZADD/ZCARD | API 제한 |

### RedisBroker 메서드 매핑

| 메서드 | Redis 명령 | API 엔드포인트 |
|--------|-----------|---------------|
| `publish()` | PUBLISH | POST `/redis/pubsub/publish` |
| `subscribe()` | SUBSCRIBE | (서버 내부 사용) |
| `stream_add()` | XADD | POST `/redis/stream/add` |
| `stream_read()` | XREAD | GET `/redis/stream/read` |
| `stream_create_group()` | XGROUP CREATE | POST `/redis/stream/group/create` |
| `stream_read_group()` | XREADGROUP + XACK | GET `/redis/stream/group/read` |
| `stream_info()` | XINFO | GET `/redis/stream/info` |
| `queue_push()` | LPUSH | POST `/redis/queue/push` |
| `queue_pop()` | BRPOP | GET `/redis/queue/pop` |
| `queue_length()` | LLEN | GET `/redis/queue/length` |
| `cache_set()` | SET EX | POST `/redis/cache/set` |
| `cache_get()` | GET + TTL | GET `/redis/cache/get/{key}` |
| `cache_delete()` | DEL | DELETE `/redis/cache/delete/{key}` |
| `rate_limit_check()` | Pipeline(ZREMRANGEBYSCORE+ZCARD+ZADD+EXPIRE) | GET `/redis/ratelimit/check` |

### 핵심 교훈

1. **Redis는 만능이 아닙니다** -- 각 패턴의 한계를 이해하고 적합한 곳에 사용
2. **Pub/Sub vs Stream** -- 유실 가능 여부가 선택의 핵심 기준
3. **Pipeline** -- 여러 명령을 원자적으로 실행할 때 반드시 사용
4. **TTL** -- 캐시는 반드시 만료 시간을 설정해야 메모리 누수 방지

### 다음 단계
- **03_rabbitmq_deep_dive.ipynb** -- RabbitMQ AMQP 프로토콜 심화 (Exchange, DLQ, Priority)
- **04_kafka_deep_dive.ipynb** -- Kafka 분산 아키텍처 심화 (Partition, Consumer Group, Batch)

In [70]:
# 정리: 연결 종료
await r.close()
await client.aclose()
print("노트북 02 완료 -- Redis 5가지 패턴 학습 끝")

노트북 02 완료 -- Redis 5가지 패턴 학습 끝


/var/folders/4c/mtw8vtqd0jdglcbdrhn5b9j00000gn/T/ipykernel_86172/1503160865.py:2: DeprecationWarning: Call to deprecated close. (Use aclose() instead) -- Deprecated since version 5.0.1.
  await r.close()


In [71]:
# 분산 Lock 실습: SET NX EX
import uuid

lock_key = "lock:order-123"
lock_id = str(uuid.uuid4())[:8]

# Lock 획득 시도 (NX = 키가 없을 때만)
acquired = await r.set(lock_key, lock_id, nx=True, ex=10)
print(f"Lock 획득: {acquired} (lock_id={lock_id})")

# 같은 키로 다시 시도 → 이미 존재하므로 실패
acquired2 = await r.set(lock_key, "other-worker", nx=True, ex=10)
print(f"중복 획득 시도: {acquired2} → NX 덕분에 차단됨")

# Lock 해제 (본인의 lock만 해제 - Lua 스크립트로 원자적 처리)
lua_release = "if redis.call('GET',KEYS[1])==ARGV[1] then return redis.call('DEL',KEYS[1]) end return 0"
released = await r.eval(lua_release, 1, lock_key, lock_id)
print(f"Lock 해제: {released} (1=성공, 0=실패)")

Lock 획득: True (lock_id=1766fcbc)
중복 획득 시도: None → NX 덕분에 차단됨
Lock 해제: 1 (1=성공, 0=실패)
